# Línea Base para Predicción de Combates Pokémon

### Importación de Módulos

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
import json
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from scipy.sparse import csr_matrix

import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

### Carga de Datos

In [7]:
df_cleaned = pd.read_csv("../data/gen9ou_cleaned_dataset.csv")
df_cleaned.head()

,battle_id,p1_poke1,p1_poke2,p1_poke3,p1_poke4,p1_poke5,p1_poke6,p2_poke1,p2_poke2,p2_poke3,p2_poke4,p2_poke5,p2_poke6,p1_win
0,1636793-gen9ou-2460221181,Dragapult,Kingambit,Hawlucha,Glimmora,Chesnaught,Gallade,Porygon-Z,Great Tusk,Raging Bolt,Corviknight,Meowscarada,Dragonite,1
1,1636793-gen9ou-2460221181,Porygon-Z,Great Tusk,Raging Bolt,Corviknight,Meowscarada,Dragonite,Dragapult,Kingambit,Hawlucha,Glimmora,Chesnaught,Gallade,0
2,95801-gen9ou-2513923983,Meganium,Arboliva,Avalugg,Ditto,Houndoom,Tinkaton,Hawlucha,Rillaboom,Azumarill,Iron Moth,Corviknight,Raging Bolt,1
3,95801-gen9ou-2513923983,Hawlucha,Rillaboom,Azumarill,Iron Moth,Corviknight,Raging Bolt,Meganium,Arboliva,Avalugg,Ditto,Houndoom,Tinkaton,0
4,1514712-gen9ou-2429668049,Dragonite,Zamazenta,LandorusTherianTherian,Gholdengo,Darkrai,Hatterene,Ninetales,Ceruledge,Venusaur,Hatterene,Great Tusk,Walking Wake,1


## 1. Separación de Conjuntos de Entrenamiento y Prueba

Split según battle_id para evitar data leakage

In [8]:
battle_ids = df_cleaned['battle_id'].unique()

train_val_ids, test_ids = train_test_split(
    battle_ids,
    test_size=0.2,
    random_state=RANDOM_STATE
)

train_ids, val_ids = train_test_split(
    train_val_ids,
    test_size=0.2, 
    random_state=RANDOM_STATE
)

train_df = df_cleaned[df_cleaned["battle_id"].isin(train_ids)]
val_df   = df_cleaned[df_cleaned["battle_id"].isin(val_ids)]
test_df  = df_cleaned[df_cleaned["battle_id"].isin(test_ids)]

Se guardan en .csv las batallas pertenecientes a cada conjunto para su posterior uso

In [9]:
pd.DataFrame({"battle_id": train_ids}).to_csv("../splits/train_ids.csv", index=False)
pd.DataFrame({"battle_id": val_ids}).to_csv("../splits/val_ids.csv", index=False)
pd.DataFrame({"battle_id": test_ids}).to_csv("../splits/test_ids.csv", index=False)

## 2. Encoding

In [10]:
pokemon_cols = [
    "p1_poke1","p1_poke2","p1_poke3","p1_poke4","p1_poke5","p1_poke6",
    "p2_poke1","p2_poke2","p2_poke3","p2_poke4","p2_poke5","p2_poke6"
]

all_pokemon = pd.unique(df_cleaned[pokemon_cols].values.ravel())

pokemon_to_idx = {p: i for i, p in enumerate(sorted(all_pokemon))}
n_pokemon = len(pokemon_to_idx)

print("Número de Pokémon:", n_pokemon)

Número de Pokémon: 809


Se codifica el dataset utilizando Multi-hot encoding (manual) con una representación de 1 para cada Pokémon presente en el combate y 0 para los ausentes. Esto se hace para ambos equipos (equipo 1 y equipo 2) y se concatenan las representaciones para formar la matriz de características final.

Además, se utiliza `scipy.sparse.csr_matrix` para convertir la matriz de características a un formato disperso, lo que es eficiente en términos de memoria dado que la mayoría de los valores serán ceros.

In [11]:
def build_sparse_matrix(df, pokemon_to_idx):
    n_rows = len(df)
    n_poke_unique = len(pokemon_to_idx)

    p1_cols = ["p1_poke1","p1_poke2","p1_poke3","p1_poke4","p1_poke5","p1_poke6"]
    p2_cols = ["p2_poke1","p2_poke2","p2_poke3","p2_poke4","p2_poke5","p2_poke6"]

    p1_idx = np.stack([df[c].map(pokemon_to_idx).values for c in p1_cols], axis=1)
    p2_idx = np.stack([df[c].map(pokemon_to_idx).values for c in p2_cols], axis=1)

    # Offset para el jugador 2
    p2_idx = p2_idx + n_poke_unique

    # Cada fila tendrá 12 índices (6 de p1 y 6 de p2)
    all_indices = np.hstack([p1_idx, p2_idx]) 

    # 'rows' debe repetirse 12 veces por cada fila del DF
    rows = np.repeat(np.arange(n_rows), 12)
    cols = all_indices.flatten()
    data = np.ones(len(rows), dtype=np.int8)

    X = csr_matrix(
        (data, (rows, cols)),
        shape=(n_rows, 2 * n_poke_unique),
        dtype=np.int8
    )

    y = df["p1_win"].to_numpy()
    return X, y

In [12]:
X_train, y_train = build_sparse_matrix(train_df, pokemon_to_idx)
X_val, y_val     = build_sparse_matrix(val_df, pokemon_to_idx)
X_test, y_test   = build_sparse_matrix(test_df, pokemon_to_idx)

print(X_train.shape)

(2255146, 1618)


A continuación, se selecciona una muestra de 100000 combates para entrenar el modelo de baseline de Random Forest, debido a limitaciones de memoria y tiempo de entrenamiento.

In [13]:
RF_SAMPLE_BATTLES = 100_000

rf_battle_ids = rng.choice(
    train_df["battle_id"].unique(),
    size=RF_SAMPLE_BATTLES,
    replace=False
)

rf_df = train_df[train_df["battle_id"].isin(rf_battle_ids)]

In [14]:
X_rf_train, y_rf_train = build_sparse_matrix(rf_df, pokemon_to_idx)

## 3. Entrenamiento del Modelo de Baseline

### 3.2 Entrenamiento de Random Forest

Ya que Random Forest no puede manejar matrices dispersas, convertimos a formato denso (aunque esto puede consumir mucha memoria, por lo que se recomienda hacerlo solo con una muestra de los datos).

In [15]:
X_rf_train_dense = X_rf_train.toarray()
X_val_dense = X_val.toarray()
X_test_dense = X_test.toarray()

Se define a continuación el la función objetivo para Optuna, que busca los mejores hiperparámetros para el modelo de Random Forest utilizando la métrica F1-score para evaluar el rendimiento del modelo durante la optimización.

In [16]:
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier

def rf_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 600),
        "max_depth": trial.suggest_int("max_depth", 5, 25), 
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        "class_weight": "balanced",
        "n_jobs": -1,
        "random_state": RANDOM_STATE
    }
    
    model = RandomForestClassifier(**params)
    model.fit(X_rf_train, y_rf_train)

    # Probabilidades (clave para ROC-AUC)
    val_proba = model.predict_proba(X_val)[:, 1]

    # Métrica independiente de threshold
    roc_auc = roc_auc_score(y_val, val_proba)

    return roc_auc

In [17]:
study_name = "optuna_rf_classic_baseline"

study_rf = optuna.create_study(
    study_name=study_name,
    direction="maximize",
    storage="sqlite:///../studies/" + study_name + ".db",
    load_if_exists=True,
)

study_rf.optimize(rf_objective, n_trials=5)


best_params = study_rf.best_params

with open(f"../parameters/{study_name}.json", "w") as f:
    json.dump(best_params, f, indent=4)

[I 2026-04-13 12:52:18,346] Using an existing study with name 'optuna_rf_classic_baseline' instead of creating a new one.
[I 2026-04-13 12:53:11,573] Trial 30 finished with value: 0.5519415238286003 and parameters: {'n_estimators': 378, 'max_depth': 20, 'min_samples_split': 9, 'min_samples_leaf': 3, 'max_features': 'sqrt'}. Best is trial 18 with value: 0.5535638991348764.
[I 2026-04-13 12:53:53,861] Trial 31 finished with value: 0.5523674412782154 and parameters: {'n_estimators': 241, 'max_depth': 23, 'min_samples_split': 8, 'min_samples_leaf': 3, 'max_features': 'sqrt'}. Best is trial 18 with value: 0.5535638991348764.
[I 2026-04-13 12:54:34,679] Trial 32 finished with value: 0.5528301713994157 and parameters: {'n_estimators': 253, 'max_depth': 23, 'min_samples_split': 8, 'min_samples_leaf': 4, 'max_features': 'sqrt'}. Best is trial 18 with value: 0.5535638991348764.
[I 2026-04-13 12:55:18,215] Trial 33 finished with value: 0.5518051865309654 and parameters: {'n_estimators': 315, 'max

In [18]:
best_params = study_rf.best_params.copy()

rf_model = RandomForestClassifier(
    **best_params,
    n_jobs=-1,
    random_state=RANDOM_STATE
)

rf_model.fit(X_rf_train_dense, y_rf_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",535
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",24
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",9
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",5
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

Tras el entrenamiento con los mejores hiperparámetros, se evalúa el modelo en el conjunto de prueba.

In [19]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    log_loss,
    confusion_matrix
)

val_proba = rf_model.predict_proba(X_val_dense)[:, 1]


thresholds = np.linspace(0.3, 0.7, 100)

best_thr = 0.5
best_f1_macro = 0

for thr in thresholds:
    preds = (val_proba >= thr).astype(int)
    f1_macro = f1_score(y_val, preds, average="macro")
    
    if f1_macro > best_f1_macro:
        best_f1_macro = f1_macro
        best_thr = thr

print(f"Best threshold (F1 macro): {best_thr:.3f}")

Best threshold (F1 macro): 0.498


In [20]:


test_proba = rf_model.predict_proba(X_test_dense)[:, 1]

test_preds = (test_proba >= best_thr).astype(int)

acc = accuracy_score(y_test, test_preds)
f1_macro = f1_score(y_test, test_preds, average="macro")
roc_auc = roc_auc_score(y_test, test_proba)
ll = log_loss(y_test, test_proba)
cm = confusion_matrix(y_test, test_preds)

print("\n=== RESULTADOS EN TEST ===")
print(f"Accuracy:  {acc:.4f}")
print(f"F1 macro:  {f1_macro:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")
print(f"Log Loss:  {ll:.4f}")

print("\nMatriz de confusión:")
print(cm)


=== RESULTADOS EN TEST ===
Accuracy:  0.5373
F1 macro:  0.5360
ROC-AUC:   0.5547
Log Loss:  0.6904

Matriz de confusión:
[[170518 181849]
 [144197 208170]]


Para comparar métricas, se incluye dos baseline naive que predice siempre la clase mayoritaria del conjunto de entrenamiento, y otro que predice aleatoriamente según la distribución de clases del conjunto de entrenamiento.

In [21]:
majority_class = np.bincount(y_train).argmax()
y_pred_naive = np.full_like(y_test, majority_class)
y_pred_random = np.random.randint(0, 2, size=len(y_test))


print("Naive Accuracy:", accuracy_score(y_test, y_pred_naive))
print("Naive F1 macro:", f1_score(y_test, y_pred_naive, average="macro"))
print("Random Accuracy:", accuracy_score(y_test, y_pred_random))
print("Random F1 macro:", f1_score(y_test, y_pred_random, average="macro"))

Naive Accuracy: 0.5
Naive F1 macro: 0.3333333333333333
Random Accuracy: 0.49995884972202276
Random F1 macro: 0.4999581426820823


### 3.3 Entrenamiento de LightGBM

Dado que LightGBM puede manejar matrices dispersas, se entrena directamente con la matriz en formato CSR sin necesidad de convertir a formato denso, lo que es más eficiente en términos de memoria. Por lo tanto, se puede entrenar con un mayor número de muestras sin preocuparse por el consumo de memoria.

In [22]:
# Convertir los conjuntos de entrenamiento, validación y prueba
X_train_float = X_train.astype('float32')
X_val_float = X_val.astype('float32')
X_test_float = X_test.astype('float32')

# Nota: y_train, y_val, y_test suelen estar bien como int o float, 
# pero si quieres asegurar compatibilidad total:
y_train_float = y_train.astype('float32')
y_val_float = y_val.astype('float32')
y_test_float = y_test.astype('float32')

Por compatibilidad con LightGBM, se convierte el tipo de datos de las características a `float32`.

In [23]:
import lightgbm as lgb

def lgb_objective(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1200),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15),
        "num_leaves": trial.suggest_int("num_leaves", 16, 256),
        "max_depth": trial.suggest_int("max_depth", -1, 30),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 200),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "n_jobs": -1,
        "random_state": RANDOM_STATE,
        "objective": "binary",
        "metric": "auc"
    }


    model = lgb.LGBMClassifier(**params)

    model.fit(
        X_train_float,
        y_train_float,
        eval_set=[(X_val_float, y_val_float)],
        eval_metric="auc",
        callbacks=[
            lgb.early_stopping(50),
            lgb.log_evaluation(0)
        ]
    )

    val_proba = model.predict_proba(X_val_float)[:,1]

    roc_auc = roc_auc_score(y_val_float, val_proba)

    return roc_auc

Definimos la función objetivo de Optuna para optimizar los hiperparámetros de LightGBM, además de escoger el mejor threshold para convertir las probabilidades de salida en predicciones binarias. Se utiliza la métrica F1-score para evaluar el rendimiento del modelo durante la optimización.

In [24]:
study_name = "optuna_lgb_classic_baseline"

study = optuna.create_study(
    study_name=study_name,
    direction="maximize",
    storage="sqlite:///../studies/" + study_name + ".db",
    load_if_exists=True
)

study.optimize(lgb_objective, n_trials=10)

best_params = study.best_params

with open(f"../parameters/{study_name}.json", "w") as f:
    json.dump(best_params, f, indent=4)

print("Best params:", study.best_params)
print("Best score:", study.best_value)

[I 2026-04-13 13:05:53,967] Using an existing study with name 'optuna_lgb_classic_baseline' instead of creating a new one.


[LightGBM] [Info] Number of positive: 1127573, number of negative: 1127573
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 2.111896 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2461
[LightGBM] [Info] Number of data points in the train set: 2255146, number of used features: 1230
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Training until validation scores don't improve for 50 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive 

[I 2026-04-13 13:08:08,884] Trial 20 finished with value: 0.5887435798633832 and parameters: {'n_estimators': 954, 'learning_rate': 0.08937367256424332, 'num_leaves': 241, 'max_depth': 10, 'min_child_samples': 46, 'subsample': 0.9503814925263468, 'colsample_bytree': 0.9565683956872199}. Best is trial 12 with value: 0.6004682670076119.


[LightGBM] [Info] Number of positive: 1127573, number of negative: 1127573
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.923128 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1949
[LightGBM] [Info] Number of data points in the train set: 2255146, number of used features: 974
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Training until validation scores don't improve for 50 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with

[I 2026-04-13 13:10:38,692] Trial 21 finished with value: 0.598483581086844 and parameters: {'n_estimators': 880, 'learning_rate': 0.10184355289313962, 'num_leaves': 173, 'max_depth': 24, 'min_child_samples': 198, 'subsample': 0.9549791962520994, 'colsample_bytree': 0.8586569405482338}. Best is trial 12 with value: 0.6004682670076119.


...

[LightGBM] [Info] Number of positive: 1127573, number of negative: 1127573
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.632334 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2003
[LightGBM] [Info] Number of data points in the train set: 2255146, number of used features: 1001
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 50 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits wit

[I 2026-04-13 13:29:57,006] Trial 28 finished with value: 0.5814991662571158 and parameters: {'n_estimators': 1193, 'learning_rate': 0.10819514784146424, 'num_leaves': 238, 'max_depth': 6, 'min_child_samples': 163, 'subsample': 0.7555634605067645, 'colsample_bytree': 0.9667029363284455}. Best is trial 27 with value: 0.6018114169344717.


[LightGBM] [Info] Number of positive: 1127573, number of negative: 1127573
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.877526 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2185
[LightGBM] [Info] Number of data points in the train set: 2255146, number of used features: 1092
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Training until validation scores don't improve for 50 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive 

[I 2026-04-13 13:32:58,622] Trial 29 finished with value: 0.5953963446720052 and parameters: {'n_estimators': 1074, 'learning_rate': 0.11315352909552073, 'num_leaves': 254, 'max_depth': 11, 'min_child_samples': 91, 'subsample': 0.6936287282948264, 'colsample_bytree': 0.9995812085028585}. Best is trial 27 with value: 0.6018114169344717.


Best params: {'n_estimators': 1198, 'learning_rate': 0.11165260825267398, 'num_leaves': 235, 'max_depth': 27, 'min_child_samples': 165, 'subsample': 0.8474973707652133, 'colsample_bytree': 0.8847324759331013}
Best score: 0.6018114169344717


Una vez obtenidos los mejores hiperparámetros, se entrena el modelo final de LightGBM con esos parámetros y se evalúa su rendimiento en el conjunto de test.

In [25]:
best_model = lgb.LGBMClassifier(**study.best_params)

best_model.fit(
    X_train_float,
    y_train_float,
    eval_set=[(X_val_float, y_val_float)],
    eval_metric="auc",
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)]
)

val_proba = best_model.predict_proba(X_val_float)[:, 1]

[LightGBM] [Info] Number of positive: 1127573, number of negative: 1127573
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 2.888064 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2014
[LightGBM] [Info] Number of data points in the train set: 2255146, number of used features: 1007
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Training until validation scores don't improve for 50 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive 

Una vez entrenado el modelo, se ajusta el threshold en función de F1-score macro.

In [26]:
thresholds = np.linspace(0.3, 0.7, 100)

best_thr = 0.5
best_f1_macro = 0

for thr in thresholds:
    preds = (val_proba >= thr).astype(int)
    f1_macro = f1_score(y_val, preds, average="macro")
    
    if f1_macro > best_f1_macro:
        best_f1_macro = f1_macro
        best_thr = thr

print(f"Best threshold (F1 macro): {best_thr:.3f}")

Best threshold (F1 macro): 0.498


In [27]:
test_proba = best_model.predict_proba(X_test_float)[:, 1]

test_preds = (test_proba >= best_thr).astype(int)

acc = accuracy_score(y_test_float, test_preds)
f1_macro = f1_score(y_test_float, test_preds, average="macro")
roc_auc = roc_auc_score(y_test_float, test_proba)
ll = log_loss(y_test_float, test_proba)
cm = confusion_matrix(y_test_float, test_preds)

print("\n=== RESULTADOS EN TEST ===")
print(f"Accuracy:  {acc:.4f}")
print(f"F1 macro:  {f1_macro:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")
print(f"Log Loss:  {ll:.4f}")

print("\nMatriz de confusión:")
print(cm)


=== RESULTADOS EN TEST ===
Accuracy:  0.5705
F1 macro:  0.5704
ROC-AUC:   0.6012
Log Loss:  0.6766

Matriz de confusión:
[[198081 154286]
 [148417 203950]]


## 4. Representación Team Differential

Se basa en la idea de representar cada combate como la diferencia entre los pokémon del equipo 1 y los pokémon del equipo 2. Esto se hace restando la representación de los pokémon del equipo 2 de la representación de los pokémon del equipo 1 de la forma `team_diff_vector = team1_vector - team2_vector`, lo que da como resultado una nueva representación que captura las diferencias entre ambos equipos. 

In [28]:
def build_team_differential(df, pokemon_to_idx):
    n_pokemon = len(pokemon_to_idx)
    
    rows = []
    cols = []
    data = []
    
    y = df['p1_win'].values
    
    for row_idx, row in enumerate(df.itertuples(index=False)):
        # team 1: +1
        for p in [row.p1_poke1,row.p1_poke2,row.p1_poke3,row.p1_poke4,row.p1_poke5,row.p1_poke6]:
            cols.append(pokemon_to_idx[p])
            rows.append(row_idx)
            data.append(1)
        
        # team 2: -1
        for p in [row.p2_poke1,row.p2_poke2,row.p2_poke3,row.p2_poke4,row.p2_poke5,row.p2_poke6]:
            cols.append(pokemon_to_idx[p])
            rows.append(row_idx)
            data.append(-1)
    
    X = csr_matrix((data, (rows, cols)), shape=(len(df), n_pokemon), dtype=np.int8)
    
    return X, y

Se introduce el experimento de eliminar la duplicación de datos, ya que con esta representación no es necesario invertir los equipos para evitar el sesgo del orden, dado que la diferencia ya captura esa información.

In [47]:
# =========================
# ELIMINAR DUPLICADOS POR BATTLE_ID en X e y
# =========================

def remove_battle_duplicates(df):
    print(f"Original size: {len(df)}")
    
    df_unique = df.drop_duplicates(subset="battle_id", keep="first").copy()
    
    print(f"After removing duplicates: {len(df_unique)}")
    print(f"Removed: {len(df) - len(df_unique)} rows")
    
    return df_unique


# Aplicar a cada split
train_df_nodup = remove_battle_duplicates(train_df)
val_df_nodup   = remove_battle_duplicates(val_df)
test_df_nodup  = remove_battle_duplicates(test_df)

Original size: 2255146
After removing duplicates: 1127573
Removed: 1127573 rows
Original size: 563788
After removing duplicates: 281894
Removed: 281894 rows
Original size: 704734
After removing duplicates: 352367
Removed: 352367 rows


In [48]:
# Train completo
X_train_diff, y_train = build_team_differential(train_df, pokemon_to_idx)
X_test_diff, y_test = build_team_differential(test_df, pokemon_to_idx)
X_val_diff, y_val = build_team_differential(val_df, pokemon_to_idx)

#hay que meter el conjunto de validación también
#INTERESANTE PROBAR SI CON ESTA REPRESENTACION ES NECESARIA LA DUPLICACION

print("Team Differential shapes:")
print("X_train:", X_train_diff.shape)
print("X_test :", X_test_diff.shape)
print("X_val  :", X_val_diff.shape)

Team Differential shapes:
X_train: (2255146, 809)
X_test : (704734, 809)
X_val  : (563788, 809)


Al igual que antes, se selecciona una muestra de 100000 combates para entrenar el modelo de baseline con esta nueva representación.

In [49]:
RF_SAMPLE_BATTLES = 100_000  # ajustar según memoria

rf_battle_ids = rng.choice(
    train_df["battle_id"].unique(),
    size=RF_SAMPLE_BATTLES,
    replace=False
)

rf_df = train_df[train_df["battle_id"].isin(rf_battle_ids)]

X_rf_diff, y_rf = build_team_differential(rf_df, pokemon_to_idx)

# Random Forest necesita matriz densa
X_train_rf_diff_dense = X_rf_diff.toarray()
X_test_rf_diff_dense = X_test_diff.toarray()
X_val_rf_diff_dense = X_val_diff.toarray()

## 5. Entrenamiento de Modelos con Representación Team Differential

### 5.1 Entrenamiento de Random Forest con Team Differential

In [50]:
def rf_objective_diff(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 600),
        "max_depth": trial.suggest_int("max_depth", 5, 25), 
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        "class_weight": "balanced",
        "n_jobs": -1,
        "random_state": RANDOM_STATE
    }
    
    model = RandomForestClassifier(**params)
    model.fit(X_train_rf_diff_dense, y_rf_train)

    # Probabilidades (clave para ROC-AUC)
    val_proba = model.predict_proba(X_val_rf_diff_dense)[:, 1]

    # Métrica independiente de threshold
    roc_auc = roc_auc_score(y_val, val_proba)

    return roc_auc

In [51]:
print("y_train unique:", np.unique(y_rf))
print("y_val unique:", np.unique(y_val))

y_train unique: [0 1]
y_val unique: [0 1]


In [52]:
study_name = "optuna_rf_diff_baseline"

study_rf = optuna.create_study(
    study_name=study_name,
    direction="maximize",
    storage="sqlite:///../studies/" + study_name + ".db",
    load_if_exists=True,
)

study_rf.optimize(rf_objective_diff, n_trials=5)

best_params = study_rf.best_params

with open(f"../parameters/{study_name}.json", "w") as f:
    json.dump(best_params, f, indent=4)

[I 2026-04-13 14:17:46,438] Using an existing study with name 'optuna_rf_diff_baseline' instead of creating a new one.
[I 2026-04-13 14:20:16,482] Trial 14 finished with value: 0.5514791829328549 and parameters: {'n_estimators': 593, 'max_depth': 20, 'min_samples_split': 7, 'min_samples_leaf': 1, 'max_features': 'log2'}. Best is trial 14 with value: 0.5514791829328549.
[I 2026-04-13 14:22:22,321] Trial 15 finished with value: 0.5513709191165537 and parameters: {'n_estimators': 582, 'max_depth': 19, 'min_samples_split': 7, 'min_samples_leaf': 1, 'max_features': 'log2'}. Best is trial 14 with value: 0.5514791829328549.
[I 2026-04-13 14:24:51,235] Trial 16 finished with value: 0.5529661767300647 and parameters: {'n_estimators': 511, 'max_depth': 25, 'min_samples_split': 7, 'min_samples_leaf': 1, 'max_features': 'log2'}. Best is trial 16 with value: 0.5529661767300647.
[I 2026-04-13 14:26:46,484] Trial 17 finished with value: 0.5511035206501105 and parameters: {'n_estimators': 495, 'max_de

In [53]:
best_params = study_rf.best_params.copy()

rf_model_diff = RandomForestClassifier(
    **best_params,
    n_jobs=-1,
    random_state=RANDOM_STATE
)

rf_model_diff.fit(X_train_rf_diff_dense, y_rf_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",511
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",25
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",7
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'log2'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

In [54]:
val_proba = rf_model_diff.predict_proba(X_val_rf_diff_dense)[:, 1]

thresholds = np.linspace(0.3, 0.7, 100)

best_thr = 0.5
best_f1_macro = 0

for thr in thresholds:
    preds = (val_proba >= thr).astype(int)
    f1_macro = f1_score(y_val, preds, average="macro")
    
    if f1_macro > best_f1_macro:
        best_f1_macro = f1_macro
        best_thr = thr

print(f"Best threshold (F1 macro): {best_thr:.3f}")

Best threshold (F1 macro): 0.498


In [55]:
test_proba = rf_model_diff.predict_proba(X_test_rf_diff_dense)[:, 1]

test_preds = (test_proba >= best_thr).astype(int)

acc = accuracy_score(y_test, test_preds)
f1_macro = f1_score(y_test, test_preds, average="macro")
roc_auc = roc_auc_score(y_test, test_proba)
ll = log_loss(y_test, test_proba)
cm = confusion_matrix(y_test, test_preds)

print("\n=== RESULTADOS EN TEST ===")
print(f"Accuracy:  {acc:.4f}")
print(f"F1 macro:  {f1_macro:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")
print(f"Log Loss:  {ll:.4f}")

print("\nMatriz de confusión:")
print(cm)


=== RESULTADOS EN TEST ===
Accuracy:  0.5356
F1 macro:  0.5312
ROC-AUC:   0.5536
Log Loss:  0.6914

Matriz de confusión:
[[154871 197496]
 [129805 222562]]


Se almacena tanto el modelo final como el metadata con el mejor threshold encontrado.

In [56]:
model_name = "rf_team_diff_best_baseline"

joblib.dump(rf_model_diff, f"../models/{model_name}.joblib")

print(f"Modelo guardado en: ../models/{model_name}.joblib")

metadata = {
    "model": "RandomForest",
    "representation": "team_differential",
    "dataset": "final",
    "params": best_params,
    "metric": "roc_auc",
    "best_threshold": float(best_thr),
    "threshold_metric": "f1_macro",
    "threshold_range": [0.3, 0.7]
}

with open(f"../models/{model_name}.json", "w") as f:
    json.dump(metadata, f, indent=4)

Modelo guardado en: ../models/rf_team_diff_best_baseline.joblib


### 5.2 Entrenamiento de LightGBM con Team Differential

In [57]:
X_train_diff = X_train_diff.astype('float32')
X_test_diff = X_test_diff.astype('float32')
X_val_diff = X_val_diff.astype('float32')

In [58]:
import lightgbm as lgb

def lgb_objective_diff(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1200),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15),
        "num_leaves": trial.suggest_int("num_leaves", 16, 256),
        "max_depth": trial.suggest_int("max_depth", -1, 30),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 200),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "n_jobs": -1,
        "random_state": RANDOM_STATE,
        "objective": "binary",
        "metric": "auc"
    }


    model = lgb.LGBMClassifier(**params)

    model.fit(
        X_train_diff,
        y_train,
        eval_set=[(X_val_diff, y_val)],
        eval_metric="auc",
        callbacks=[
            lgb.early_stopping(50),
            lgb.log_evaluation(0)
        ]
    )

    val_proba = model.predict_proba(X_val_diff)[:,1]

    roc_auc = roc_auc_score(y_val, val_proba)

    return roc_auc

In [59]:
study_name = "optuna_lgb_diff_baseline"

study = optuna.create_study(
    study_name=study_name,
    direction="maximize",
    storage="sqlite:///../studies/" + study_name + ".db",
    load_if_exists=True
)

study.optimize(lgb_objective_diff, n_trials=10)

best_params = study.best_params

with open(f"../parameters/{study_name}.json", "w") as f:
    json.dump(best_params, f, indent=4)

print("Best params:", study.best_params)
print("Best score:", study.best_value)

[I 2026-04-13 14:32:05,492] Using an existing study with name 'optuna_lgb_diff_baseline' instead of creating a new one.


[LightGBM] [Info] Number of positive: 1127573, number of negative: 1127573
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.158653 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1969
[LightGBM] [Info] Number of data points in the train set: 2255146, number of used features: 659
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Training until validation scores don't improve for 50 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive g

[I 2026-04-13 14:34:46,833] Trial 20 finished with value: 0.5907393306309456 and parameters: {'n_estimators': 988, 'learning_rate': 0.11716103705243436, 'num_leaves': 123, 'max_depth': 19, 'min_child_samples': 44, 'subsample': 0.9435927985073209, 'colsample_bytree': 0.8312881172408886}. Best is trial 14 with value: 0.5933463640308269.


[LightGBM] [Info] Number of positive: 1127573, number of negative: 1127573
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.036596 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1744
[LightGBM] [Info] Number of data points in the train set: 2255146, number of used features: 581
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Training until validation scores don't improve for 50 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive g

[I 2026-04-13 14:37:51,561] Trial 21 finished with value: 0.5925456173148106 and parameters: {'n_estimators': 1064, 'learning_rate': 0.08533882171400291, 'num_leaves': 166, 'max_depth': 25, 'min_child_samples': 81, 'subsample': 0.871775453294046, 'colsample_bytree': 0.9964411031904936}. Best is trial 14 with value: 0.5933463640308269.


...

[LightGBM] [Info] Number of positive: 1127573, number of negative: 1127573
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.046759 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1618
[LightGBM] [Info] Number of data points in the train set: 2255146, number of used features: 539
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Training until validation scores don't improve for 50 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Did not meet early stopping. Best iteration is:
[1089]	valid_0's auc: 0.59405


[I 2026-04-13 14:58:27,981] Trial 28 finished with value: 0.5940500312336039 and parameters: {'n_estimators': 1105, 'learning_rate': 0.12339016184435486, 'num_leaves': 206, 'max_depth': 27, 'min_child_samples': 118, 'subsample': 0.7204453598219356, 'colsample_bytree': 0.9273132565498566}. Best is trial 27 with value: 0.5952488187045635.


[LightGBM] [Info] Number of positive: 1127573, number of negative: 1127573
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.926322 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1537
[LightGBM] [Info] Number of data points in the train set: 2255146, number of used features: 512
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Training until validation scores don't improve for 50 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with

[I 2026-04-13 15:02:02,336] Trial 29 finished with value: 0.5951431837755901 and parameters: {'n_estimators': 1097, 'learning_rate': 0.12356371908988825, 'num_leaves': 254, 'max_depth': 27, 'min_child_samples': 167, 'subsample': 0.7224173449637258, 'colsample_bytree': 0.9327690628786354}. Best is trial 27 with value: 0.5952488187045635.


Best params: {'n_estimators': 1087, 'learning_rate': 0.11992888700918213, 'num_leaves': 255, 'max_depth': 26, 'min_child_samples': 116, 'subsample': 0.8095345937689584, 'colsample_bytree': 0.9287779489376592}
Best score: 0.5952488187045635


In [60]:
best_model_lgbm = lgb.LGBMClassifier(**study.best_params)

best_model_lgbm.fit(
    X_train_diff,
    y_train,
    eval_set=[(X_val_diff, y_val)],
    eval_metric="auc",
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)]
)

val_proba = best_model_lgbm.predict_proba(X_val_diff)[:, 1]

[LightGBM] [Info] Number of positive: 1127573, number of negative: 1127573
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.402410 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1617
[LightGBM] [Info] Number of data points in the train set: 2255146, number of used features: 539
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Training until validation scores don't improve for 50 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive g

In [61]:
thresholds = np.linspace(0.3, 0.7, 100)

best_thr = 0.5
best_f1_macro = 0

for thr in thresholds:
    preds = (val_proba >= thr).astype(int)
    f1_macro = f1_score(y_val, preds, average="macro")
    
    if f1_macro > best_f1_macro:
        best_f1_macro = f1_macro
        best_thr = thr

print(f"Best threshold (F1 macro): {best_thr:.3f}")

Best threshold (F1 macro): 0.502


In [62]:
test_proba = best_model.predict_proba(X_test_float)[:, 1]

test_preds = (test_proba >= best_thr).astype(int)

acc = accuracy_score(y_test_float, test_preds)
f1_macro = f1_score(y_test_float, test_preds, average="macro")
roc_auc = roc_auc_score(y_test_float, test_proba)
ll = log_loss(y_test_float, test_proba)
cm = confusion_matrix(y_test_float, test_preds)

print("\n=== RESULTADOS EN TEST ===")
print(f"Accuracy:  {acc:.4f}")
print(f"F1 macro:  {f1_macro:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")
print(f"Log Loss:  {ll:.4f}")

print("\nMatriz de confusión:")
print(cm)


=== RESULTADOS EN TEST ===
Accuracy:  0.5705
F1 macro:  0.5705
ROC-AUC:   0.6012
Log Loss:  0.6766

Matriz de confusión:
[[204000 148367]
 [154313 198054]]


Se almacena tanto el modelo final como el metadata con el mejor threshold encontrado.

In [63]:
model_name = "lgb_team_diff_best_baseline"

joblib.dump(best_model_lgbm, f"../models/{model_name}.joblib")

print(f"Modelo guardado en: ../models/{model_name}.joblib")

metadata = {
    "model": "LightGBM",
    "representation": "team_differential",
    "dataset": "final",
    "params": best_params,
    "metric": "roc_auc",
    "best_threshold": float(best_thr),
    "threshold_metric": "f1_macro",
    "threshold_range": [0.3, 0.7]
}

with open(f"../models/{model_name}.json", "w") as f:
    json.dump(metadata, f, indent=4)

Modelo guardado en: ../models/lgb_team_diff_best_baseline.joblib


## 6. Conclusiones

Ya que las obtenidas con la representación de Team Differential son iguales a las obtenidas con la representación original, pero genera menos columnas, se concluye que la representación de Team Differential es más eficiente y no pierde información relevante para la predicción, por lo que en adelante se utilizará esta representación.

Sobre duplicación de datos, al eliminarse, el modelo obtiene un sesgo claro al predecir, al solo existir una sola clase en el conjunto y, por lo tanto aunque se guarden los dataset sin duplicados, no se podrán entrenar modelos con esta representación, por lo que se descarta eliminar los duplicados.

In [43]:
##eliminar la duplicación de battle_id en df_cleaned y guardarlo en un csv llamado gen9ou_cleaned_no_duplicates.csv

df_cleaned_actual_metagame = pd.read_csv("../data/gen9ou_actual_metagame_03-26_cleaned_dataset.csv")

df_cleaned_no_duplicates = df_cleaned.drop_duplicates(subset=['battle_id'])
df_cleaned_no_duplicates_actual_metagame = df_cleaned_actual_metagame.drop_duplicates(subset=['battle_id'])

df_cleaned_no_duplicates.to_csv("../data/gen9ou_cleaned_no_duplicates.csv", index=False)
df_cleaned_no_duplicates_actual_metagame.to_csv("../data/gen9ou_cleaned_no_duplicates_actual_metagame.csv", index=False)

print(f"Tamaño original: {len(df_cleaned)}")
print(f"Tamaño sin duplicados: {len(df_cleaned_no_duplicates)}")
print(f"Tamaño sin duplicados (actual metagame): {len(df_cleaned_no_duplicates_actual_metagame)}")

KeyboardInterrupt: 